# Installations and Imported Packages

In [1]:
#!pip install wandb

In [2]:
import numpy as np
import pandas as pd
import os
import random
from kaggle_secrets import UserSecretsClient
import wandb
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import torch
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

# User Defined Control Variable

Setting the flag variable **ignore_milestone_solutions_code** as **False** ignore the exection of the cells containing the milestone solution code

In [3]:
ignore_milestone_solutions_code=False

# Weights & Biases Setup and Test run

In [4]:
# try:
#     wandb.login()
#     print("Login Successful")
# except Exception as e:
#     print(f"Login Failed: {e}")

In [5]:
# for run in range(5):
#     # Start a new wandb run to track this script.
#     run = wandb.init(
#         # Set the wandb entity where your project will be logged (generally your team name).
#         entity="21f2000660-dl-gen-ai-project-26-t1",
#         # Set the wandb project where this run will be logged.
#         project="21f2000660-t12026",
#         # Track hyperparameters and run metadata.
#         config={
#             "learning_rate": 0.02,
#             "architecture": "CNN",
#             "dataset": "CIFAR-100",
#             "epochs": 10,
#         },
#     )

# # Simulate training.
# epochs = 10
# offset = random.random() / 5
# for epoch in range(2, epochs):
#     acc = 1 - 2**-epoch - random.random() / epoch - offset
#     loss = 2**-epoch + random.random() / epoch + offset

#     # Log metrics to wandb.
#     run.log({"acc": acc, "loss": loss})

# # Finish the run and upload any remaining data.
# run.finish()

# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [6]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [7]:
# CONFIGURATION
path = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
DATA_ROOT = path # Enter dataset path

with os.scandir(path) as entries:
    directories = [entry.name for entry in entries]

GENRES = sorted(directories) # Make the list of all genres available (alphabetical order)
print(GENRES)

STEMS = {'drums':'drums.wav', 'vocals':'vocals.wav', 'bass':'bass.wav', 'other':'other.wav'} # Write here stems file name
print(STEMS)

STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
{'drums': 'drums.wav', 'vocals': 'vocals.wav', 'bass': 'bass.wav', 'other': 'other.wav'}


In [8]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    #print(train_dataset)

    rng = random.Random(seed)

    # ------------------- write your code here -------------------------------

        # Iterate through Genres
        # Check: if genre folder exists
        # CHECK : Completeness (Does it have all stems?)
        # CHECK : Corruption (Is any file too small? (less than 4kb))
        # size checks
        # Stratified Shuffle Split
     #-------------------------------------------------------------------------

    # Limits given in the questions
    limit_1 = 4 * 1024
    limit_2 = 5.0491 * 1024 * 1024
    limit_3 = 5.0493 * 1024 * 1024

    corrupted_songs = 0
    small_songs = 0
    large_songs = 0
    
    # Helper function to populate dict
    def add_to_dict(target_dict, genre, song_paths):
        for song_path in song_paths:
            for stem_key, stem_file in STEMS.items():
                target_dict[genre][stem_key].append(os.path.join(song_path, stem_file))

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        
        # Check: if genre folder exists
        if not os.path.isdir(genre_path):
            continue
            
        # Get all song folders
        song_folders = sorted([f.path for f in os.scandir(genre_path) if f.is_dir()])
        valid_songs = []

        #print(song_folders[:3])

        for song in song_folders:
            is_valid = True
            
            # All stems for a song should be available, if not invalid 
            for stem_file in STEMS.values():
                file_path = os.path.join(song, stem_file)
                
                if not os.path.exists(file_path):
                    is_valid = False
                    continue
                
                file_size = os.path.getsize(file_path)
                
                if file_size < limit_1:
                    corrupted_songs += 1
                    is_valid = False
                
                if file_size < limit_2:
                    small_songs += 1
                    
                if file_size > limit_3:
                    large_songs += 1

            if is_valid:
                valid_songs.append(song)

        # Stratified Shuffle Split
        rng.shuffle(valid_songs)
        
        split_index = int(len(valid_songs) * (1 - val_split))
        train_paths = valid_songs[:split_index]
        val_paths = valid_songs[split_index:]
        
        add_to_dict(train_dataset, genre, train_paths)
        add_to_dict(val_dataset, genre, val_paths)

    print('Q1',corrupted_songs + small_songs)
    print('Q2',abs(large_songs - small_songs))
    print('Q3',abs(len(train_dataset['reggae']['drums']) - len(val_dataset['country']['vocals'])))

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

#print(val[GENRE_TO_TEST])

Q1 1256
Q2 1072
Q3 66


In [9]:
if ignore_milestone_solutions_code:
    
    def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
        """
        Input:
            dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
        Output:
            df: Pandas DataFrame containing details of all files with silence >= 5s
        """
        records = []
        # ------------------- write your code here -------------------------------
    
        total_files = 0    # ---- COUNT TOTAL FILES ----
    
    
    
            # Load Audio
    
            # Find Non-Silent Intervals
    
            # CASE A: Fully silent
            # CASE B: START silence
            # CASE C: END silence
            # CASE D: MIDDLE silence
    
            # Store result
            # if max_silence >= threshold_sec:
            #     records.append({
            #         "Genre": genre,
            #         "Stem": stem_name,
            #         "Duration": round(total_duration, 2),
            #         "Max_Silence_Sec": round(max_silence, 2),
            #         "Silence_Location": ", ".join(silence_type),
            #         "File_Path": file_path
            #     })
        #-------------------------------------------------------------------------
    
        # Iterate through the nested dictionary
        for genre, stems in dataset_dict.items():
            for stem_name, file_paths in stems.items():
                for file_path in file_paths:
                    total_files += 1
                    
                    # Load Audio
                    try:
                        y, sr = librosa.load(file_path, sr=sr)
                        total_duration = librosa.get_duration(y=y, sr=sr)
                    except Exception as e:
                        print(f"Error loading {file_path}: {e}")
                        continue
    
                    # Find Non-Silent Intervals
                    # intervals is a list of [start, end] sample indices of NON-silence
                    intervals = librosa.effects.split(y, top_db=top_db)
                    
                    silence_durations = []
                    silence_type = []
    
                    # CASE A: Fully silent
                    if len(intervals) == 0:
                        max_silence = total_duration
                        silence_type.append("Full")
                    else:
                        # CASE B: START silence (from 0 to first non-silent start)
                        if intervals[0][0] > 0:
                            start_silence = intervals[0][0] / sr
                            silence_durations.append(start_silence)
                            if start_silence >= threshold_sec:
                                 silence_type.append("Start")
    
                        # CASE C: END silence (from last non-silent end to total length)
                        if intervals[-1][1] < len(y):
                            end_silence = (len(y) - intervals[-1][1]) / sr
                            silence_durations.append(end_silence)
                            if end_silence >= threshold_sec:
                                 silence_type.append("End")
    
                        # CASE D: MIDDLE silence (gaps between intervals)
                        for i in range(len(intervals) - 1):
                            silence_gap = (intervals[i+1][0] - intervals[i][1]) / sr
                            silence_durations.append(silence_gap)
                            if silence_gap >= threshold_sec:
                                 silence_type.append("Middle")
                        
                        # Determine max silence for this file
                        if silence_durations:
                            max_silence = max(silence_durations)
                        else:
                            max_silence = 0.0
    
                    # Store result
                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 4),
                            "Silence_Location": ", ".join(set(silence_type)), # use set to avoid duplicates
                            "File_Path": file_path
                        })
        df = pd.DataFrame(records)
        return df
    
    
    # --- EXECUTION ---
    # Pass your 'tr' (training) dictionary here.
    # Ensure 'tr' is defined from your previous build_dataset code.
    df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)
    
    # --- RESULTS ANALYSIS ---
    
    # ------------------- write your code here -------------------------------
    
    print('Q4',len(df_silence))
    print('Q5',len(df_silence[df_silence['Stem'] == 'vocals']))
    print('Q6',df_silence[df_silence['Stem'] == 'vocals']['Max_Silence_Sec'].mean())
    print('Q7',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]))
    print('Q8',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums') & (df_silence['Silence_Location'] == 'Middle')]))
    print('Q9',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums') & (df_silence['Max_Silence_Sec'] >= 10.0)]))
    #-------------------------------------------------------------------------
    # Hint: Create a pivot Table: Count by Genre vs Stem
    
    pivot_table = df_silence.pivot_table(index='Genre', columns='Stem', values='Max_Silence_Sec', aggfunc='count', fill_value=0)
    print("\n--- Pivot Table ---")
    print(pivot_table)

In [10]:
if ignore_milestone_solutions_code:
    
    stems_audio = []
    try:
        for key in STEM_KEYS:
        # ------------------- write your code here -------------------------------
        # Load audio (Duration 5.0s for speed/consistency)
            file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
    
            y, sr = librosa.load(file_path, sr=SR, duration=DURATION)
            stems_audio.append(y)
        #-------------------------------------------------------------------------
    
        print("Audio loaded successfully.")
    except NameError:
        print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
    except IndexError:
        print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
    except Exception as e:
        print(f"ERROR: {e}")

In [11]:
if ignore_milestone_solutions_code:
    
    # ------------------- write your code here -------------------------------
    # Stack them into a numpy array (Shape: 4 x Samples)
    stems_stack = np.vstack(stems_audio)
    
    # Mix the stems by summing them element-wise
    mix_raw = np.sum(stems_stack, axis=0)
    
    # Calculate RMS Amplitude MANUALLY
    rms_val = np.sqrt(np.mean(mix_raw**2))
    
    #Peak Normalization
    max_val = np.max(np.abs(mix_raw))
    
    if max_val > 0:
        mix_norm = mix_raw / max_val
    else:
        mix_norm = mix_raw
    
    # VALIDATION
    assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
    #------------------------------------------------------------------------
    
    print('Q10',len(mix_raw))
    print('Q11',rms_val)
    print('Q12',max_val)

# Milestone 2

In [12]:
if ignore_milestone_solutions_code:
    
    #Mean duration of Jazz stems

    jazz_durations = []
    for stem_dict in tr.get('jazz').values():
        for file_path in stem_dict:
            dur = librosa.get_duration(path=file_path)
            jazz_durations.append(dur)
    
    mean_jazz_dur = np.mean(jazz_durations)
    print('Q1',mean_jazz_dur)

In [13]:
if ignore_milestone_solutions_code:

    #Unique sample rates
    
    unique_srs = set()
    for genre, stems in tr.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                file_sr = librosa.get_samplerate(file_path)
                unique_srs.add(file_sr)
    
    print('Q2', unique_srs)

In [14]:
if ignore_milestone_solutions_code:
    
    #Corrupted or zero-byte files
    
    zero_byte_count = 0
    for genre, stems in tr.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                if os.path.getsize(file_path) == 0:
                    zero_byte_count += 1
    
    print('Q3',zero_byte_count)

In [15]:
if ignore_milestone_solutions_code:
    
    #Average peak amplitude for vocal stems
    
    vocal_peak_dbs = []
    for genre, stems in tr.items():
        vocal_files = stems.get('vocals')
        for file_path in vocal_files:
            y, sr = librosa.load(file_path, sr=SR)
            peak_amp = np.max(np.abs(y))
            peak_db = 20 * np.log10(max(peak_amp, 1e-10))
            vocal_peak_dbs.append(peak_db)
    
    avg_vocal_peak_db = np.mean(vocal_peak_dbs)
    print('Q4',avg_vocal_peak_db)

In [16]:
if ignore_milestone_solutions_code:
    
    #Spectral Centroids
    
    genre_mean_centroids = {}
    
    for genre, stems in tr.items():
        genre_centroids = []
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                y, _ = librosa.load(file_path, sr=SR)
                # Compute spectral centroid
                sc = librosa.feature.spectral_centroid(y=y, sr=SR)
                # Take the mean centroid of this specific file
                genre_centroids.append(np.mean(sc))
                
        if genre_centroids:
            genre_mean_centroids[genre] = np.mean(genre_centroids)
    
    print(genre_mean_centroids)
    
    blues_centroid = genre_mean_centroids.get('blues')
    print('Q5',blues_centroid)
    
    highest_sc_genre = max(genre_mean_centroids, key=genre_mean_centroids.get)
    print('Q6',highest_sc_genre)

In [17]:
if ignore_milestone_solutions_code:
    
    #Silence in the first 0.5 seconds
    silence_count = 0
    for genre, stems in tr.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                y_start, sr = librosa.load(file_path, sr=SR, duration=0.5)
                
                intervals = librosa.effects.split(y_start, top_db=TOP_DB)
                
                if len(intervals) == 0 or intervals[0][0] > 0:
                    silence_count += 1
    
    print('Q7',silence_count)

# Programming Question

Complete the code as instructed in comments and answer the questions that follow.

In [18]:
# --- 1. Setup and Preprocessing ---
ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

def extract_features(song_path):
    # Load 10s at 22050Hz
    y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    return [float(tempo), spec_cent, zcr, rolloff]

# --- 2. Data Preparation & Stratified Split ---
data = []
for g in GENRES:
    gp = os.path.join(STEMS_PATH, g)
    songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
    for s in songs[:50]: # Sampling 50 for speed; use all for final
        data.append({'path': os.path.join(gp, s), 'genre': g})

df = pd.DataFrame(data)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)

# --- 3. Model Training (Decision Tree) ---
X_train = np.array([extract_features(p) for p in train_df['path']])
y_train = train_df['genre']
X_val = np.array([extract_features(p) for p in val_df['path']])
y_val = val_df['genre']

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

'''
YOUR CODE HERE

y_pred = # COMPUTE PREDICTED VALUES
macro_f1 = # COMPUTE VALIDATION MACRO F1 SCORE
cm = # COMPUTE CONFUSION MATRIX
cr = # COMPUTE CLASSIFICATION REPORT

'''

y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')
cm = confusion_matrix(y_val, y_pred, labels=GENRES)
cr = classification_report(y_val, y_pred, target_names=GENRES)

print(f"Validation Macro F1 Score: {macro_f1:.4f}\n")
print("Detailed Classification Report:")
print(cr)

Validation Macro F1 Score: 0.1523

Detailed Classification Report:
              precision    recall  f1-score   support

       blues       0.20      0.10      0.13        10
   classical       0.14      0.10      0.12        10
     country       0.10      0.10      0.10        10
       disco       0.20      0.40      0.27        10
      hiphop       0.25      0.10      0.14        10
        jazz       0.00      0.00      0.00        10
       metal       0.41      0.90      0.56        10
         pop       0.20      0.20      0.20        10
      reggae       0.00      0.00      0.00        10
        rock       0.00      0.00      0.00        10

    accuracy                           0.19       100
   macro avg       0.15      0.19      0.15       100
weighted avg       0.15      0.19      0.15       100



In [19]:
if ignore_milestone_solutions_code:
    
    '''
    YOUR CODE HERE
    
    Visualize the confusion matrix and compute TP, TN, FP, FN for all genres.
    '''
    # Visualize Confusion Matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=GENRES, yticklabels=GENRES)
    plt.title('Confusion Matrix: Decision Tree')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    print('\n')
    print('--- TP, TN, FP, FN for all genres  ---')
    for i,genre in enumerate(GENRES):
        tp=cm[i, i]
        fp=cm[:, i].sum()-tp
        fn=cm[i, :].sum()-tp
        tn=cm.sum()-(tp+fp+fn)
            
        print(genre,"\t\t",tp,tn,fp,fn)

# DummyModel Submission for Registration

In [20]:
if ignore_milestone_solutions_code:
    
    testData = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
    path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
    
    with os.scandir(path) as entries:
        directories = [entry.name for entry in entries]
    
    ids = [f"{i:04d}" for i in range(1, len(testData)+1)]
    genres = random.choices(directories, k=3020)
    
    submission = pd.DataFrame({
        'id': ids,
        'genre' : genres
    })
    
    submission.head()
    
    #submission.to_csv('/kaggle/working/submission.csv', index=False)

# Submission using the DecisionTreeClassifier model

In [21]:
TEST_CSV_PATH = os.path.join(ROOT, 'test.csv')

def extract_features(song_path):
    # Load 10s at 22050Hz
    y, sr = librosa.load(song_path, sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    return [float(tempo), spec_cent, zcr, rolloff]

test_df = pd.read_csv(TEST_CSV_PATH)

#print(test_df.shape)
#print(test_df.sample(5))

X_test = []

for index, row in test_df.iterrows():
    file_name = row['filename']
    file_path = os.path.join(ROOT, file_name)
    features = extract_features(file_path)
    X_test.append(features)

X_test = np.array(X_test)
test_pred = clf.predict(X_test)

submission = pd.DataFrame({
    'id': test_df['id'],
    'genre' : test_pred
})

submission.head()

submission.to_csv('/kaggle/working/submission.csv', index=False)